# Bolo 04: launch video narration (Indic Parler TTS)

Needs the `HF_TOKEN` secret. GPU T4. Output: `/kaggle/working/narration/<id>.wav` (best take) and `takes/`, plus `scores.json`.

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git soundfile jiwer

In [ ]:
import os
os.environ["USE_TF"] = "0"; os.environ["USE_TORCH"] = "1"; os.environ["USE_JAX"] = "0"; os.environ["USE_FLAX"] = "0"

In [ ]:
import json, os
WORK = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'

# Indic Parler-TTS is gated: accept its terms on huggingface.co first, then add a READ token as a secret
# named HF_TOKEN (Kaggle: Add-ons -> Secrets; Colab: key icon -> Secrets).
token, errs = None, []
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    errs.append(f"kaggle: {type(e).__name__}: {e}")
if not token and not os.path.exists("/kaggle"):
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        errs.append(f"colab: {type(e).__name__}: {e}")
token = token or os.environ.get("HF_TOKEN")
if token:
    from huggingface_hub import login
    login(token=token)
    MODEL = "ai4bharat/indic-parler-tts"
else:
    # API-started runs cannot read Kaggle secrets: run a pipeline SMOKE TEST with the ungated Parler-TTS mini
    # (same codebase/architecture) and stop. The real run must be started from the Kaggle UI.
    print("NO HF_TOKEN:", " | ".join(errs), "-> smoke-test mode with parler-tts/parler-tts-mini-v1")
    MODEL = "parler-tts/parler-tts-mini-v1"
SMOKE = MODEL != "ai4bharat/indic-parler-tts"

In [ ]:
SPEC = {
 "voices": {
  "narrator": "Mary speaks in a warm, clear, confident voice at a moderate pace, with very clear audio and no background noise.",
  "hinglish": "Divya speaks in a friendly, natural voice at a moderate pace, with very clear audio and no background noise."
 },
 "lines": [
  {
   "id": "01_hook",
   "voice": "hinglish",
   "text": "docker चालू करो. volume चालीस पे कर दो."
  },
  {
   "id": "02_problem",
   "voice": "narrator",
   "text": "This is how millions of people in India actually talk to their computers. Hindi and English, mixed in one sentence. Most voice tools get lost."
  },
  {
   "id": "03_intro",
   "voice": "narrator",
   "text": "Bolo is a small, open toolkit that understands Hinglish voice commands. It runs entirely on your own laptop. No internet, no cloud, and it costs nothing."
  },
  {
   "id": "04_sun",
   "voice": "narrator",
   "text": "First, Bolo Sun, a speech model of about one hundred and forty megabytes, writes down what you said."
  },
  {
   "id": "05_tiny",
   "voice": "narrator",
   "text": "Then Bolo Tiny, smaller than one megabyte, works out what you want, and which folder, app, or contact you mean."
  },
  {
   "id": "06_ground",
   "voice": "narrator",
   "text": "Finally, Bolo checks your machine. Does that folder really exist? If it is not sure, it does not guess. It asks."
  },
  {
   "id": "07_demo",
   "voice": "narrator",
   "text": "Here it is, running on a MacBook Air."
  },
  {
   "id": "08_numbers",
   "voice": "narrator",
   "text": "On two hundred and forty test commands, spoken in computer voices it never trained on, Bolo picked the right action eighty two percent of the time. The Hinglish speech model it started from got fifty nine percent on its own, and sixty nine percent even with Bolo's smarter decoding."
  },
  {
   "id": "10_limits",
   "voice": "narrator",
   "text": "It is an early preview. It has been tested on only a few real voices, and on long, general speech it is a little worse than the model it started from. We would rather tell you that up front."
  },
  {
   "id": "11_close",
   "voice": "narrator",
   "text": "Bolo. Open source, and free. Try it, and tell us where it fails."
  }
 ]
}

In [ ]:
import torch, soundfile as sf, numpy as np, jiwer, re
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, pipeline
assert not SMOKE, "needs the gated Indic Parler model: start this run from the Kaggle UI with the HF_TOKEN secret"
dev = "cuda"
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL).to(dev)
tok = AutoTokenizer.from_pretrained(MODEL)
desc_tok = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
asr = pipeline("automatic-speech-recognition", "openai/whisper-small", device=0)
SR = model.config.sampling_rate
OUT = WORK + "/narration"; os.makedirs(OUT + "/takes", exist_ok=True)
norm = lambda s: re.sub(r"[^a-z0-9ऀ-ॿ ]", " ", s.lower()).split()
scores = {}
for line in SPEC["lines"]:
    d = desc_tok(SPEC["voices"][line["voice"]], return_tensors="pt").to(dev)
    p = tok(line["text"], return_tensors="pt").to(dev)
    takes = []
    for seed in range(3):
        torch.manual_seed(seed)
        with torch.no_grad():
            a = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                               prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask, do_sample=True)
        a = a.cpu().float().numpy().squeeze()
        f = f"{OUT}/takes/{line['id']}_s{seed}.wav"
        sf.write(f, a, SR)
        lang = "hi" if line["voice"] == "hinglish" else "en"
        hyp = asr(f, generate_kwargs={"language": lang, "task": "transcribe"})["text"]
        w = jiwer.wer(" ".join(norm(line["text"])) or "x", " ".join(norm(hyp)) or "x") if lang == "en" else None
        takes.append({"seed": seed, "wer": w, "dur": round(len(a) / SR, 2), "heard": hyp})
        print(line["id"], seed, w, round(len(a) / SR, 2), hyp[:90], flush=True)
    best = min(takes, key=lambda t: (t["wer"] if t["wer"] is not None else 0, abs(t["dur"] - len(line["text"]) / 14)))
    os.replace(f"{OUT}/takes/{line['id']}_s{best['seed']}.wav", f"{OUT}/{line['id']}.wav")
    scores[line["id"]] = {"best": best, "takes": takes}
json.dump({"sr": SR, "scores": scores}, open(OUT + "/scores.json", "w"), indent=1, ensure_ascii=False)